# FashionCGAN — Laporan Visual

Notebook ini **tidak melakukan training**. Semua proses berjalan lokal lewat CLI, lalu notebook membaca artefak di `outputs/` untuk membuat gambar laporan dan slide presentasi.

## Alur kerja

```bash
# 1. Training (sekali; ~13-15 menit di CPU)
python -m fashion_cgan.train --config configs/default.yaml --device cpu

# 2. Evaluasi dengan classifier
python -m fashion_cgan.evaluate outputs/cgan_epoch_50.pt --output-dir outputs/evaluation
```

## Isi notebook

1. Setup: konfigurasi run, cek artefak wajib.
2. Dataset asli: contoh per kelas + distribusi kelas.
3. Kurva loss training.
4. Perkembangan kualitas generator antar-epoch (tabel PNG).
5. Grid sampel per kelas dari generasi terakhir.
6. Keragaman sampel dalam satu kelas.
7. Evaluasi classifier: akurasi asli vs akurasi per kelas generator.
8. Ringkasan untuk laporan + rekomendasi slide PPT.

> Semua path relatif dari root repo. Jalankan notebook dari root agar `data/` dan `outputs/` terbaca.

## 1. Setup

Ubah `RUN_DIR` bila memakai hasil run yang berbeda.

Semua artefak dihasilkan lokal; notebook ini tidak memakai Colab.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from fashion_cgan.data import get_dataloaders
from fashion_cgan.generate import generate_grid
from fashion_cgan.models import Generator
from fashion_cgan.utils import FASHION_MNIST_CLASSES, denormalize, load_metrics, save_image_grid

RUN_DIR = ROOT / "outputs"
EVAL_DIR = ROOT / "outputs" / "evaluation"
FIG_DIR = ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

checkpoints = sorted(RUN_DIR.glob("cgan_epoch_*.pt"), key=lambda p: int(p.stem.rsplit("_", 1)[1]))
CHECKPOINT = checkpoints[-1]
metrics_path = RUN_DIR / "metrics.csv"

for required in (CHECKPOINT, metrics_path):
    assert required.exists(), f"artefak tidak ditemukan: {required}; jalankan training dulu"

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
print(f"checkpoint : {CHECKPOINT.name}")
print(f"jumlah ckpt: {len(checkpoints)}")
print(f"device     : {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 2. Dataset asli

Contoh satu gambar per kelas dan distribusi jumlah sampel per kelas. Berguna untuk slide "dataset".

In [ ]:
train_loader, test_loader = get_dataloaders(root=ROOT / "data", batch_size=128, download=False)
train_set = train_loader.dataset

first_index = {}
for index in range(len(train_set)):
    label = int(train_set[index][1])
    first_index.setdefault(label, index)

fig, axes = plt.subplots(2, 5, figsize=(11, 4.6))
for label, ax in enumerate(axes.flat):
    image, _ = train_set[first_index[label]]
    ax.imshow(denormalize(image).squeeze(), cmap="gray")
    ax.set_title(f"{label}: {FASHION_MNIST_CLASSES[label]}", fontsize=9)
    ax.axis("off")
fig.suptitle("Fashion-MNIST - satu contoh per kelas", fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "dataset_samples.png", bbox_inches="tight")
plt.show()

In [ ]:
targets = torch.tensor([int(train_set[i][1]) for i in range(len(train_set))])
counts = torch.bincount(targets, minlength=10).tolist()

fig, ax = plt.subplots(figsize=(7.5, 3.2))
bars = ax.bar(FASHION_MNIST_CLASSES, counts, color="#4C72B0")
ax.bar_label(bars, fontsize=8)
ax.set_ylabel("jumlah sampel")
ax.set_title("Distribusi kelas training Fashion-MNIST")
plt.xticks(rotation=35, ha="right")
fig.tight_layout()
fig.savefig(FIG_DIR / "class_distribution.png", bbox_inches="tight")
plt.show()

print("total train:", sum(counts), "| min:", min(counts), "| max:", max(counts))

## 3. Kurva loss training

Loss adversarial tidak mengukur kualitas gambar, tetapi menunjukkan keseimbangan D dan G. Idealnya keduanya stabil di sekitar nilai yang tidak ekstrem.

In [ ]:
rows = load_metrics(metrics_path)
epochs = [row["epoch"] for row in rows]
d_loss = [row["discriminator_loss"] for row in rows]
g_loss = [row["generator_loss"] for row in rows]

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(epochs, d_loss, label="Discriminator", color="#C44E52")
ax.plot(epochs, g_loss, label="Generator", color="#4C72B0")
ax.set_xlabel("epoch")
ax.set_ylabel("BCEWithLogitsLoss")
ax.set_title("Kurva loss training cGAN")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "loss_curve.png", bbox_inches="tight")
plt.show()

print(f"epoch tercatat: {len(rows)}")
print(f"D awal/akhir : {d_loss[0]:.3f} -> {d_loss[-1]:.3f}")
print(f"G awal/akhir : {g_loss[0]:.3f} -> {g_loss[-1]:.3f}")

## 4. Perkembangan kualitas antar-epoch

Grid 10 kelas dari beberapa checkpoint. Cocok untuk slide "generator makin baik seiring epoch". Noise yang sama dipakai di semua epoch agar perbandingan adil.

In [ ]:
def load_generator(checkpoint_path: Path) -> Generator:
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    config = checkpoint["config"]
    generator = Generator(config["model"]["latent_dim"], config["data"]["classes"])
    generator.load_state_dict(checkpoint["generator"])
    return generator.eval()


epochs_to_show = sorted({1, len(checkpoints) // 2, len(checkpoints)}) + (
    [rows[-1]["epoch"]] if rows[-1]["epoch"] not in {1, len(checkpoints) // 2, len(checkpoints)} else []
)
epochs_to_show = [e for e in epochs_to_show if 1 <= e <= len(checkpoints)]
labels = list(range(10))

torch.manual_seed(42)
fixed_noise = torch.randn(len(labels), 100)
label_tensor = torch.arange(len(labels))

fig, axes = plt.subplots(len(labels), len(epochs_to_show), figsize=(1.35 * len(epochs_to_show) + 1, 14))
for column, epoch in enumerate(epochs_to_show):
    generator = load_generator(RUN_DIR / f"cgan_epoch_{epoch}.pt")
    with torch.inference_mode():
        images = generator(fixed_noise, label_tensor)
    for label, ax in enumerate(axes[:, column]):
        ax.imshow(denormalize(images[label]).squeeze(), cmap="gray")
        ax.axis("off")
        if column == 0:
            ax.set_ylabel(str(label), rotation=0, labelpad=12, fontsize=9)
    axes[0, column].set_title(f"epoch {epoch}", fontsize=10)
fig.suptitle("Perkembangan Generator per epoch (noise tetap)", fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "progression_by_epoch.png", bbox_inches="tight")
plt.show()
print("epoch ditampilkan:", epochs_to_show)

## 5. Sampel per kelas (generasi terakhir)

Empat sampel berbeda per kelas dari checkpoint terakhir. Sekaligus menyimpan grid PNG untuk laporan.

In [ ]:
generator = load_generator(CHECKPOINT)
samples_per_class = 4

fig, axes = plt.subplots(10, samples_per_class, figsize=(samples_per_class * 1.5 + 1, 13))
with torch.inference_mode():
    for label, ax_row in enumerate(axes):
        noise = torch.randn(samples_per_class, 100, generator=torch.Generator().manual_seed(label))
        images = generator(noise, torch.full((samples_per_class,), label))
        for ax, image in zip(ax_row, images):
            ax.imshow(denormalize(image).squeeze(), cmap="gray")
            ax.axis("off")
        ax_row[0].set_ylabel(FASHION_MNIST_CLASSES[label], rotation=0, labelpad=60, fontsize=9, va="center")
fig.suptitle(f"Sampel per kelas - {CHECKPOINT.stem}", fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "samples_per_class.png", bbox_inches="tight")
plt.show()

generate_grid(CHECKPOINT, list(range(10)), FIG_DIR / "grid_one_per_class.png", seed=42)

## 6. Keragaman sampel dalam satu kelas

Bila semua sampel hampir identik, generator mengalami mode collapse meski loss terlihat normal.

In [ ]:
def sample_class(label: int, count: int = 12, seed: int = 0) -> torch.Tensor:
    noise = torch.randn(count, 100, generator=torch.Generator().manual_seed(seed))
    with torch.inference_mode():
        return generator(noise, torch.full((count,), label))


show_classes = [0, 3, 6, 7]
fig, axes = plt.subplots(len(show_classes), 12, figsize=(14, 5.4))
for row, label in enumerate(show_classes):
    images = sample_class(label)
    for ax, image in zip(axes[row], images):
        ax.imshow(denormalize(image).squeeze(), cmap="gray")
        ax.axis("off")
    axes[row, 0].set_ylabel(FASHION_MNIST_CLASSES[label], rotation=0, labelpad=60, fontsize=9, va="center")
fig.suptitle("Keragaman sampel per kelas", fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / "diversity_per_class.png", bbox_inches="tight")
plt.show()

## 7. Evaluasi classifier

Membutuhkan `outputs/evaluation/evaluation.json` dari `fashion_cgan.evaluate`.

- **Akurasi data asli**: classifier pada test set Fashion-MNIST.
- **Akurasi per kelas**: proporsi gambar generator yang diklasifikasikan sesuai label permintaan.

In [ ]:
report_path = EVAL_DIR / "evaluation.json"
evaluation = json.loads(report_path.read_text(encoding="utf-8")) if report_path.exists() else None

if evaluation is None:
    print("evaluation.json belum ada; jalankan:")
    print("python -m fashion_cgan.evaluate outputs/cgan_epoch_50.pt --output-dir outputs/evaluation")
else:
    names = list(evaluation["generated"])
    scores = [evaluation["generated"][name]["accuracy"] for name in names]
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(names, scores, color="#55A868")
    ax.axhline(evaluation["real_accuracy"], color="#C44E52", linestyle="--", label="akurasi data asli")
    ax.bar_label(bars, fmt="%.2f", fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("akurasi")
    ax.set_title(f"Kesesuaian gambar generator per kelas (epoch {evaluation['epoch']})")
    ax.legend(frameon=False)
    plt.xticks(rotation=35, ha="right")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "classifier_accuracy.png", bbox_inches="tight")
    plt.show()

    print(f"akurasi data asli      : {evaluation['real_accuracy']:.4f}")
    print(f"rata-rata generator    : {evaluation['generated_accuracy_mean']:.4f}")
    weakest = min(names, key=lambda n: evaluation['generated'][n]['accuracy'])
    print(f"kelas terlemah         : {weakest} ({evaluation['generated'][weakest]['accuracy']:.2f})")

## 8. Ringkasan untuk laporan dan PPT

Tabel angka ringkas yang bisa langsung disalin ke laporan. Gambar tersimpan di `outputs/figures/`.

In [ ]:
summary = {
    "checkpoint": CHECKPOINT.name,
    "epochs_trained": len(checkpoints),
    "final_discriminator_loss": round(d_loss[-1], 4),
    "final_generator_loss": round(g_loss[-1], 4),
    "real_accuracy": evaluation["real_accuracy"] if evaluation else None,
    "generated_accuracy_mean": evaluation["generated_accuracy_mean"] if evaluation else None,
}
(FIG_DIR / "summary.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
print(json.dumps(summary, indent=2))

print("\nGambar laporan:")
for path in sorted(FIG_DIR.glob("*.png")):
    print(f"- {path.relative_to(ROOT)}")

### Ekspor ringkasan Markdown

File `outputs/figures/summary.md` siap ditempel ke laporan/PowerPoint.

In [ ]:
lines = [
    "# Ringkasan FashionCGAN",
    "",
    f"- Checkpoint: `{summary['checkpoint']}`",
    f"- Epoch tercatat: {summary['epochs_trained']}",
    f"- Loss D akhir: {summary['final_discriminator_loss']}",
    f"- Loss G akhir: {summary['final_generator_loss']}",
]
if evaluation:
    lines += [
        f"- Akurasi classifier pada data asli: {evaluation['real_accuracy']:.4f}",
        f"- Rata-rata akurasi generator: {evaluation['generated_accuracy_mean']:.4f}",
        "",
        "| Kelas | Akurasi |",
        "|---|---|",
    ]
    lines += [f"| {n} | {evaluation['generated'][n]['accuracy']:.2f} |" for n in evaluation["generated"]]
lines += ["", "## Gambar", ""] + [f"- `{p.name}`" for p in sorted(FIG_DIR.glob("*.png"))]
(FIG_DIR / "summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("\n".join(lines))

### Usulan urutan slide PPT

1. **Judul & pembagian kerja** — tiga anggota, area masing-masing.
2. **Latar belakang**: apa itu conditional GAN dan mengapa Fashion-MNIST.
3. **Dataset**: `dataset_samples.png` + `class_distribution.png`.
4. **Arsitektur MLP**: Generator (noise 100 + one-hot 10 → 256 → 512 → 784, Tanh) dan Discriminator (784 + 10 → 512 → 256 → 1 logit).
5. **Kondisi training**: `BCEWithLogitsLoss`, Adam lr 2e-4, betas (0.5, 0.999), batch 128, 50 epoch, seed 42, CPU lokal.
6. **Loss**: `loss_curve.png`, plus catatan bahwa loss adversarial bukan metrik kualitas.
7. **Progres visual**: `progression_by_epoch.png`.
8. **Hasil**: `samples_per_class.png` dan `grid_one_per_class.png`.
9. **Keragaman**: `diversity_per_class.png`.
10. **Evaluasi kuantitatif**: `classifier_accuracy.png` + angka asli vs generator.
11. **Keterbatasan**: MLP tanpa struktur spasial, mode collapse parsial, classifier kecil.
12. **Kesimpulan & pengembangan**: CNN sebagai eksperimen lanjutan.